# CPBO with Branch Length

In [1]:
! pip install torch tensorflow scikit-learn pandas numpy transformers

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.3.1 -> 25.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from datetime import datetime

In [16]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B")
model.eval()

Loading checkpoint shards: 100%|██████████| 4/4 [00:08<00:00,  2.08s/it]


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (n

In [17]:
def forward_sample(prompt, top_k=2, step_length=2, n_steps=8):
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids
    
    def sample_step(input_ids, depth):
        if depth == 0:
            return [input_ids]
        
        with torch.no_grad():
            outputs = model(input_ids=input_ids)
            logits = outputs.logits[:, -1, :]
        
        probs = torch.softmax(logits, dim=-1).squeeze()
        top_k_probs, top_k_indices = torch.topk(probs, top_k)
        
        sequences = []
        for i in range(top_k):
            next_token_id = top_k_indices[i].unsqueeze(0).unsqueeze(0)
            next_input_ids = torch.cat([input_ids, next_token_id], dim=-1)
            
            for _ in range(step_length - 1):
                with torch.no_grad():
                    outputs = model(input_ids=next_input_ids)
                    logits = outputs.logits[:, -1, :]
                
                probs = torch.softmax(logits, dim=-1).squeeze()
                next_token = torch.argmax(probs).unsqueeze(0).unsqueeze(0)
                next_input_ids = torch.cat([next_input_ids, next_token], dim=-1)
            
            sequences.extend(sample_step(next_input_ids, depth - 1))
        
        return sequences
    
    sampled_sequences = sample_step(input_ids, n_steps)
    decoded_sequences = [tokenizer.decode(seq[0], skip_special_tokens=True) for seq in sampled_sequences]
    
    return decoded_sequences

In [18]:
prompt = "Once upon a time"
sampled_texts = forward_sample(prompt, top_k=4, step_length=2, n_steps=6)
for text in sampled_texts:
    print(text)

Once upon a time, there was a little girl who loved to read. She
Once upon a time, there was a little girl who loved to read and write
Once upon a time, there was a little girl who loved to read books.
Once upon a time, there was a little girl who loved to read, and
Once upon a time, there was a little girl who loved the color pink.
Once upon a time, there was a little girl who loved the color purple.
Once upon a time, there was a little girl who loved the color blue.
Once upon a time, there was a little girl who loved the color yellow.
Once upon a time, there was a little girl who loved her daddy very much
Once upon a time, there was a little girl who loved her daddy. She
Once upon a time, there was a little girl who loved her daddy so much
Once upon a time, there was a little girl who loved her daddy more than
Once upon a time, there was a little girl who loved books. She loved
Once upon a time, there was a little girl who loved books. Her parents
Once upon a time, there was a little

In [19]:
# Get current date in YYYY-MM-DD format
date_str = datetime.now().strftime("%Y-%m-%d")

# Define filename
filename = f"{date_str}-output.txt"

# Write sampled texts to the file
with open(filename, "w") as f:
    for text in sampled_texts:
        f.write(text + "\n")

print(f"Saved output to {filename}")

UnicodeEncodeError: 'charmap' codec can't encode character '\u2032' in position 34: character maps to <undefined>

In [8]:
# Define a simple prompt
prompt = "Once upon a time"
inputs = tokenizer(prompt, return_tensors="pt")
output = model.generate(**inputs, max_length=40)
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print(generated_text)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Once upon a time, a long time ago, the first man was born. His name was Adam. He was created by God out of the dust of the earth. God made a beautiful garden
